In [1]:
from langgraph.graph import StateGraph, START,END
from typing import TypedDict
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver
load_dotenv()

# model=ChatGoogleGenerativeAI(model='gemini-2.5-flash')

True

In [15]:
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
llm=HuggingFaceEndpoint(
    # repo_id="HuggingFaceH4/zephyr-7b-beta",
    # repo_id="mistralai/Mistral-7B-Instruct-v0.2",
    # repo_id="HuggingFaceH4/zephyr-7b-gemma-v0.1",
    repo_id="openai/gpt-oss-20b",
    # repo_id="openai/gpt-oss-120b",
    # repo_id="Qwen/Qwen3-4B-Instruct-2507",
    task="text-generation"
)
model=ChatHuggingFace(llm=llm)
# generator=ChatHuggingFace(llm=llm)


In [3]:
class JokeState(TypedDict):
    topic:str
    joke:str
    explanation:str

In [4]:
def generate_joke(state:JokeState):
    response=model.invoke(f"Generate a joke on topic:{state['topic']}")
    return {
        'joke':response.content
    }

In [5]:
def generate_explanation(state:JokeState):
    response=model.invoke(f"Explain this joke , Joke:{state['joke']}")

    return {
        'explanation':response.content
    }

In [6]:
graph=StateGraph(JokeState)

graph.add_node('generate_joke',generate_joke)
graph.add_node('generate_explanation',generate_explanation)

graph.add_edge(START,'generate_joke')
graph.add_edge('generate_joke','generate_explanation')
graph.add_edge('generate_explanation',END)

checkpointer=InMemorySaver()

workflow=graph.compile(checkpointer=checkpointer)

In [16]:
model.invoke("hello").content

'Hello! 👋 How can I assist you today?'

In [24]:
config1={"configurable":{"thread_id":'1'}}

workflow.invoke({'topic':'sex'},config=config1)

{'topic': 'sex',
 'joke': '',
 'explanation': 'It looks like the joke didn’t come through. Could you please share the joke you’d like me to explain? Once I have the text, I’ll break it down and explain why it’s funny.'}

In [18]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'sex', 'joke': 'Why did the couple bring a ladder to their date night?\n\nBecause they heard the chemistry was *high* and they wanted to *raise* the stakes!', 'explanation': '**The joke in a nutshell**\n\n> **Why did the couple bring a ladder to their date night?  \n>  Because they heard the chemistry was *high* and they wanted to *raise* the stakes!**\n\nIt’s a little pun that plays on two separate meanings of a few words and the literal use of a ladder. Let’s break it down word‑by‑word.\n\n'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f12d946-1bdd-66dc-8002-274a9354e472'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-04-01T06:31:14.343683+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f12d946-1325-6726-8001-f44f603090a0'}}, tasks=(), interrupts=())

In [19]:
config={"configurable":{"thread_id":"1"}}
list(workflow.get_state_history(config))

[StateSnapshot(values={'topic': 'sex', 'joke': 'Why did the couple bring a ladder to their date night?\n\nBecause they heard the chemistry was *high* and they wanted to *raise* the stakes!', 'explanation': '**The joke in a nutshell**\n\n> **Why did the couple bring a ladder to their date night?  \n>  Because they heard the chemistry was *high* and they wanted to *raise* the stakes!**\n\nIt’s a little pun that plays on two separate meanings of a few words and the literal use of a ladder. Let’s break it down word‑by‑word.\n\n'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f12d946-1bdd-66dc-8002-274a9354e472'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-04-01T06:31:14.343683+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f12d946-1325-6726-8001-f44f603090a0'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'topic': 'sex', 'joke': 'Why did the couple bring a la

In [18]:
# 1f0fd2a9-72c7-60a1-8009-80d03ab8bfad
config={"configurable":{"thread_id":"1","checkpoint_id":"1f0fd2a9-72c7-60a1-8009-80d03ab8bfad"}}
workflow.invoke(None,config=config)

{'topic': 'sex',
 'joke': ' Why don\'t some crops go out on dates? Because they\'re afraid of getting reaped! But seriously, let\'s keep things PG-13 here. Here\'s a classic one:\n\nWhy did the scarecrow win an award? Because he was outstanding in his field! (And no, that field was not full of "golden wheat" or anything like that. Let\'s keep it clean, people!)',
 'explanation': ' This joke is a play on words and revolves around the double meaning of the term "outstanding in his field." In the agricultural sense, to be "outstanding in one\'s field" means excelling in farming or cultivating a specific crop. However, the phrase also has a figurative meaning, which is to be exceptional or remarkable in any field or area of expertise. In the joke, the scarecrow is being praised for his exceptional performance in his role as a scarecrow, which is to protect the crops in the field. The humor lies in the unexpected and literal interpretation of the phrase given the context of the agricultural

In [20]:
config2={
    "configurable":{
        "thread_id":"2",
    }
}

In [23]:
workflow.invoke({
    "topic":"ai agent"
},config=config2)

{'topic': 'ai agent',
 'joke': 'Why did the AI agent bring a ladder to the data center?\n\nBecause it heard the servers were *climbing* the cloud and wanted to keep up!',
 'explanation': ''}

In [22]:
workflow.get_state(config2)

StateSnapshot(values={'topic': 'ai agent', 'joke': 'Why did the AI agent get a job as a tour guide?\n\nBecause it always knows the *best* routes and can *predict* what people will *really* want to see!', 'explanation': ''}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f12d94e-0158-6c1b-8002-263e447b05b7'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-04-01T06:34:46.311412+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f12d94d-f881-6bf5-8001-e257cc764a9b'}}, tasks=(), interrupts=())

In [25]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'ai agent', 'joke': 'Why did the AI agent bring a ladder to the data center?\n\nBecause it heard the servers were *climbing* the cloud and wanted to keep up!', 'explanation': ''}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f12d94f-c963-619d-8006-0d78c0bb3d40'}}, metadata={'source': 'loop', 'step': 6, 'parents': {}}, created_at='2026-04-01T06:35:34.130713+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f12d94f-c096-652c-8005-6f35f2521c2f'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'topic': 'ai agent', 'joke': 'Why did the AI agent bring a ladder to the data center?\n\nBecause it heard the servers were *climbing* the cloud and wanted to keep up!', 'explanation': ''}, next=('generate_explanation',), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f12d94f-c096-652c-8005-6f35f2521c2f'}}, metadata={'source': 'loop', 'st